In [ ]:
try:
    import cupy as cp
    xp = cp
    fft2 = cp.fft.fft2
    fftshift = cp.fft.fftshift
    fftfreq = cp.fft.fftfreq
    asnumpy = cp.asnumpy
    use_gpu = True
    print("Using CuPy GPU backend")
except Exception:
    import numpy as np
    xp = np
    from numpy.fft import fft2, fftshift
    from numpy.fft import fftfreq
    asnumpy = lambda x: x
    use_gpu = False
    print("Using NumPy CPU backend")

from scipy.signal import windows  # only used if numpy backend; we implement hann fallback below
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import iris
import iris.plot as iplt
import iris.coord_categorisation as coord_cat
from iris import load, Constraint
import numpy as _np  # keep a reference to CPU numpy where needed (e.g. plotting)
import pandas as pd
# from cf_units import Unit
import tqdm

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
# import palettable
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings

In [ ]:
# warnings.filterwarnings("ignore", message="Missing CF-netCDF measure variable")
# iris.FUTURE.datum_support = True
# iris.FUTURE.save_split_attrs = True

# plt.rcParams['font.size'] = 10
# plt.rcParams['font.family'] = 'sans-serif'

# def plot_map(ax, data_cube, cmap, vmin=-30, vmax=30, title=None):
#     # Plot the data
#     fill = iplt.pcolormesh(data_cube, cmap=cmap, vmin=vmin, vmax=vmax, axes=ax)

#     # Add coastline contours using NaturalEarthFeature
#     coastline = cfeature.NaturalEarthFeature(category='physical',
#                                              name='coastline',
#                                              scale='50m',
#                                              edgecolor='black',
#                                              facecolor='none',
#                                              transform=ccrs.PlateCarree())
#     ax.add_feature(coastline)

#     # Add a title if provided
#     if title:
#         ax.set_title(title)

In [ ]:
def _hann(N):
    # backend-agnostic Hann; safe for N==1
    if N <= 1:
        return xp.ones(1, dtype=float)
    n = xp.arange(N)
    return 0.5 - 0.5 * xp.cos(2 * xp.pi * n / (N - 1))

def compute_2d_psd(field, dx=1.0, dy=None, window=True):
    """
    Compute 2D PSD of a gridded field on the active backend (CuPy or NumPy).
    field: 2D array (ny, nx) - can be numpy or cupy array; will be converted to backend xp
    returns: psd2d (backend array), kx, ky (backend arrays)
    """
    if dy is None:
        dy = dx
    # move to backend
    f = xp.asarray(field, dtype=float)
    ny, nx = f.shape

    # Remove mean and linear trend
    f = f - xp.nanmean(f)
    X, Y = xp.meshgrid(xp.arange(nx), xp.arange(ny))
    A = xp.column_stack([X.ravel(), Y.ravel(), xp.ones(nx * ny)])
    coeffs, *_ = xp.linalg.lstsq(A, f.ravel(), rcond=None)
    trend = (coeffs[0] * X + coeffs[1] * Y + coeffs[2]).reshape(f.shape)
    f = f - trend

    # Fill NaNs with zeros
    # xp.nan_to_num exists for both backends
    f = xp.nan_to_num(f, nan=0.0)

    # Window
    if window:
        wx = _hann(nx)
        wy = _hann(ny)
        W = xp.sqrt(xp.outer(wy, wx))
        f = f * W
        wnorm = xp.sum(W ** 2)
    else:
        wnorm = nx * ny

    # FFT
    F = fft2(f)
    psd2d = (xp.abs(F) ** 2) / (wnorm)
    kx = fftfreq(nx, d=dx)
    ky = fftfreq(ny, d=dy)
    psd2d = fftshift(psd2d)
    kx = fftshift(kx)
    ky = fftshift(ky)
    return psd2d, kx, ky

def radial_average(psd2d, kx, ky, nbins=50):
    """
    Radially average a centered PSD2D. Operates on backend arrays.
    returns ks (backend array) and psd_radial (backend array).
    """
    KX, KY = xp.meshgrid(kx, ky)
    K = xp.sqrt(KX ** 2 + KY ** 2)
    kmax = K.max()
    bins = xp.linspace(0.0, kmax, nbins + 1)
    inds = xp.digitize(K.ravel(), bins)
    psd_flat = psd2d.ravel()
    psd_rad = xp.empty(nbins, dtype=psd_flat.dtype)
    k_centers = 0.5 * (bins[:-1] + bins[1:])
    for i in range(1, nbins + 1):
        sel = inds == i
        if xp.any(sel):
            psd_rad[i - 1] = psd_flat[sel].mean()
        else:
            psd_rad[i - 1] = xp.nan
    return k_centers, psd_rad

In [ ]:
variable = "tas"

In [ ]:
target = iris.load_cube(f"/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/270126/val_SI_small_r1/{variable}_target_SI_2009-01-01_2009-12-31.nc")

In [ ]:
SI_0 = iris.load_cube(f"/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/270126/val_SI_small_r1/{variable}_ensemble_member_0_SI_2009-01-01_2009-12-31.nc")

In [ ]:
CorrDiff_0 = iris.load_cube(f"/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/270126/val_CorrDiff_small_r1/{variable}_ensemble_member_0_CorrDiff_2009-01-01_2009-12-31.nc")

In [ ]:
UNet = iris.load_cube(f"/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/270126/val_UNet_small_r1/{variable}_ensemble_mean_UNet_2009-01-01_2009-12-31.nc")

In [ ]:
EDM_0 = iris.load_cube(f"/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/output/280126/EDM_20/{variable}_ensemble_member_0_EDM_2009-01-01_2009-01-31.nc")

In [ ]:
if 'cubes' not in globals():
    try:
        print(f"Frame width 10!")
        frame_width = 10
        cubes = [
            target[:, frame_width:-frame_width, frame_width:-frame_width],
            SI_0[:, frame_width:-frame_width, frame_width:-frame_width],
            CorrDiff_0[:, frame_width:-frame_width, frame_width:-frame_width],
            UNet[:, frame_width:-frame_width, frame_width:-frame_width],
            EDM_0[:, frame_width:-frame_width, frame_width:-frame_width],
        ]
        for cube in cubes:
            cube.coord('longitude').guess_bounds()
            cube.coord('latitude').guess_bounds()
    except NameError as e:
        raise NameError(
            "Source cubes (target, SI_0, CorrDiff_0, UNet, EDM_0) are not all defined. "
            "Load those files / run the cells that set them before running this loop."
        ) from e

In [ ]:
n_dates = 30  # number of dates to process

# Prepare lists to store results
psd_truth_all = []
psd_SI_all = []
psd_CorrDiff_all = []
psd_UNet_all = []
psd_EDM_all = []

psd_truth_rad_all = []
psd_SI_rad_all = []
psd_CorrDiff_rad_all = []
psd_UNet_rad_all = []
psd_EDM_rad_all = []

for date_index in tqdm.tqdm(range(n_dates)):
    # Compute 2D PSDs
    psd_truth, kx, ky = compute_2d_psd(cubes[0][date_index].data, dx=1, window=True)
    psd_SI, _, _      = compute_2d_psd(cubes[1][date_index].data, dx=1, window=True)
    psd_CorrDiff, _, _= compute_2d_psd(cubes[2][date_index].data, dx=1, window=True)
    psd_UNet, _, _    = compute_2d_psd(cubes[3][date_index].data, dx=1, window=True)
    psd_EDM, _, _ = compute_2d_psd(cubes[4][date_index].data, dx=1, window=True)
    
    # Store 2D PSDs if needed
    psd_truth_all.append(psd_truth)
    psd_SI_all.append(psd_SI)
    psd_CorrDiff_all.append(psd_CorrDiff)
    psd_UNet_all.append(psd_UNet)
    psd_EDM_all.append(psd_EDM)

    # Compute radial averages
    k_centers, psd_truth_rad = radial_average(psd_truth, kx, ky, nbins=60)
    _, psd_SI_rad            = radial_average(psd_SI, kx, ky, nbins=60)
    _, psd_CorrDiff_rad      = radial_average(psd_CorrDiff, kx, ky, nbins=60)
    _, psd_UNet_rad           = radial_average(psd_UNet, kx, ky, nbins=60)
    _, psd_EDM_rad  = radial_average(psd_EDM, kx, ky, nbins=60)
    
    # Store radial averages
    psd_truth_rad_all.append(psd_truth_rad)
    psd_SI_rad_all.append(psd_SI_rad)
    psd_CorrDiff_rad_all.append(psd_CorrDiff_rad)
    psd_UNet_rad_all.append(psd_UNet_rad)
    psd_EDM_rad_all.append(psd_EDM_rad)

In [ ]:
psd_truth_rad_all    = np.array(psd_truth_rad_all) 
psd_SI_rad_all       = np.array(psd_SI_rad_all)
psd_CorrDiff_rad_all = np.array(psd_CorrDiff_rad_all)
psd_UNet_rad_all     = np.array(psd_UNet_rad_all)
psd_EDM_rad_all     = np.array(psd_EDM_rad_all)

In [ ]:
psd_truth_rad_all_mean = np.mean(psd_truth_rad_all, axis=0)
psd_SI_rad_all_mean = np.mean(psd_SI_rad_all, axis=0)
psd_CorrDiff_rad_all_mean = np.mean(psd_CorrDiff_rad_all, axis=0)
psd_UNet_rad_all_mean = np.mean(psd_UNet_rad_all, axis=0)
psd_EDM_rad_all_mean = np.mean(psd_EDM_rad_all, axis=0)

In [ ]:
variable 

In [ ]:
k_centers_plot = 1/np.arange(1, k_centers.shape[0]+1)

# plot (log-log)
plt.figure(figsize=(4, 3))
plt.loglog(1/k_centers_plot, psd_truth_rad_all_mean, label='Target')   # wavelength = 1/k
plt.loglog(1/k_centers_plot, psd_SI_rad_all_mean, label='CDSI')
plt.loglog(1/k_centers_plot, psd_CorrDiff_rad_all_mean, label='CorrDiff')
plt.loglog(1/k_centers_plot, psd_UNet_rad_all_mean, label='UNet')
plt.loglog(1/k_centers_plot, psd_EDM_rad_all_mean, label='EDM')
plt.xlim(1, 1E2)
plt.ylim(1E-5, 1E5)
plt.xlabel('Wavenumber')
plt.ylabel('Power')
plt.legend()
# plt.gca().invert_xaxis()  # optional: show small scales (left) to large scales (right)
plt.savefig(f"{variable}-psd-mean.pdf", format="pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot (log-log)
# plt.figure(figsize=(4, 4))
# plt.loglog(1/k_centers, psd_truth_rad_all_mean, label='HCLIM')   # wavelength = 1/k
# plt.loglog(1/k_centers, psd_SI_rad_all_mean, label='SI')
# plt.loglog(1/k_centers, psd_CorrDiff_rad_all_mean, label='CorrDiff')
# plt.loglog(1/k_centers, psd_UNet_rad_all_mean, label='UNet')
# plt.loglog(1/k_centers, psd_EDM_rad_all_mean, label='EDM')
# plt.xlim(1, 5)
# plt.ylim(5E-2, 5E0)
# plt.xlabel('Wavenumber')
# plt.ylabel('Power')
# plt.legend()
# plt.gca().invert_xaxis()  # optional: show small scales (left) to large scales (right)
# plt.savefig("psd-2009-01-mean-high-freq.png", format="png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
np.save(f"psd_truth_rad_all_mean_{variable}.npy", psd_truth_rad_all_mean)
np.save(f"psd_SI_rad_all_mean_{variable}.npy", psd_SI_rad_all_mean)
np.save(f"psd_CorrDiff_rad_all_mean_{variable}.npy", psd_CorrDiff_rad_all_mean)
np.save(f"psd_UNet_rad_all_mean_{variable}.npy", psd_UNet_rad_all_mean)
np.save(f"psd_EDM_rad_all_mean_{variable}.npy", psd_EDM_rad_all_mean)
np.save(f"k_centers_{variable}.npy", k_centers)  # save the radial bin centers too

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def compare_psd(var1, var2, folder='.'):
    p = Path(folder)
    names = {
        'truth': f"psd_truth_rad_all_mean_{var1}.npy",
        'SI':    f"psd_SI_rad_all_mean_{var1}.npy",
        'CD':    f"psd_CorrDiff_rad_all_mean_{var1}.npy",
        'UNet':  f"psd_UNet_rad_all_mean_{var1}.npy",
        'EDM':   f"psd_EDM_rad_all_mean_{var1}.npy",
        'k':     f"k_centers_{var1}.npy",
    }
    names2 = {k: v.replace(f"_{var1}.npy", f"_{var2}.npy") for k,v in names.items()}

    def load_map(mapping):
        out = {}
        for k,fn in mapping.items():
            path = p / fn
            if not path.exists():
                raise FileNotFoundError(f"Missing file: {path}")
            out[k] = np.load(path)
        return out

    a = load_map(names)
    b = load_map(names2)

    report = {}
    for key in ['truth','SI','CD','UNet','EDM']:
        A = a[key]
        B = b[key]
        report[key] = {
            'shape_A': A.shape, 'shape_B': B.shape,
            'allclose': np.allclose(A, B, equal_nan=True),
        }
        # compare where both finite
        mask = np.isfinite(A) & np.isfinite(B)
        if mask.any():
            diff = A[mask] - B[mask]
            with np.errstate(divide='ignore', invalid='ignore'):
                rel = np.abs(diff) / (np.maximum(np.abs(A[mask]), np.abs(B[mask])) + 1e-12)
            report[key].update({
                'max_abs_diff': float(np.nanmax(np.abs(diff))),
                'mean_abs_diff': float(np.nanmean(np.abs(diff))),
                'max_rel_diff': float(np.nanmax(rel)),
                'mean_rel_diff': float(np.nanmean(rel)),
                'n_compared': int(mask.sum()),
            })
        else:
            report[key].update({'max_abs_diff': np.nan, 'mean_abs_diff': np.nan, 'n_compared': 0})

    # print summary
    print(f"Comparing {var1} vs {var2}\n")
    for k,v in report.items():
        print(f"{k}: shapes {v['shape_A']} vs {v['shape_B']}, allclose={v['allclose']}")
        print(f"   n={v['n_compared']}, max_abs={v['max_abs_diff']:.3g}, mean_abs={v['mean_abs_diff']:.3g}, max_rel={v['max_rel_diff']:.3g}")
    print()

    # quick diagnostic plots for the radial-mean PSDs (k from var1)
    k = a['k']
    mask_k = np.isfinite(k) & (k > 0)
    plt.figure(figsize=(10,4))
    plt.subplot(1,3,1)
    plt.loglog(k[mask_k], a['truth'][mask_k], label=f"{var1}-truth")
    plt.loglog(k[mask_k], b['truth'][mask_k], label=f"{var2}-truth")
    plt.legend(); plt.xlabel('k'); plt.ylabel('Power'); plt.title('truth')

    plt.subplot(1,3,2)
    diff = a['truth'] - b['truth']
    plt.semilogx(k[mask_k], diff[mask_k])
    plt.xlabel('k'); plt.title('truth diff (A-B)')

    plt.subplot(1,3,3)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = a['truth'] / (b['truth'] + 1e-12)
    plt.semilogx(k[mask_k], ratio[mask_k])
    plt.xlabel('k'); plt.title('truth ratio (A/B)')
    plt.tight_layout()
    plt.show()

    return report

report = compare_psd('tas', 'pr')